# 06_finetune_marian_en_hi

In [1]:
from transformers import MarianMTModel, MarianTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-hi-en")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-hi-en").to(device)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

pytorch_model.bin:   0%|          | 0.00/304M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/304M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [3]:
import sys
sys.path.append("/kaggle/input/datasets/kunjagarwal/en-hi-dataset")

In [6]:
from datasets import load_from_disk
dataset_filtered= load_from_disk("/kaggle/working/iitb_filtered")
print(dataset_filtered)

DatasetDict({
    train: Dataset({
        features: ['en', 'hi'],
        num_rows: 1413872
    })
    validation: Dataset({
        features: ['en', 'hi'],
        num_rows: 520
    })
    test: Dataset({
        features: ['en', 'hi'],
        num_rows: 2505
    })
})


In [7]:
SUBSET_SIZE = 180_000  # adjust as needed

train_subset = dataset_filtered["train"].shuffle(seed=42).select(range(SUBSET_SIZE))

In [8]:
val_subset = dataset_filtered["validation"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["validation"]))))
test_subset = dataset_filtered["test"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["test"]))))

In [9]:
def tokenize_batch(sources, targets, tokenizer, max_length=80):
    """
    Tokenizes a batch of source (English) and target (Hindi) sentences
    for MarianMT training. Returns input_ids, attention_mask, and labels
    with padding positions in labels set to -100 (so loss ignores them).
    """
    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Replace pad_token_id with -100 in labels, so CrossEntropyLoss ignores padding
    labels = model_inputs["labels"]
    labels[labels == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels

    return model_inputs

In [11]:
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, max_length=80):
        self.sources = sources
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = self.sources[idx]
        target = self.targets[idx]
        tokenized = tokenize_batch([source], [target], self.tokenizer, self.max_length)
        return {key: val.squeeze(0) for key, val in tokenized.items()}

In [16]:
train_dataset = TranslationDataset(train_subset["hi"], train_subset["en"], tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [17]:
val_dataset = TranslationDataset(val_subset["hi"], val_subset["en"], tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [24]:
sample_batch = tokenize_batch([train_subset["hi"][0]], [train_subset["en"][0]], tokenizer)
print(sample_batch["labels"])

tensor([[14236,  4479,  2936,  4857,  1036, 12944,  7751,     0,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100]])


In [18]:
batch = next(iter(train_dataloader))
print(batch["input_ids"].shape)

torch.Size([16, 80])


In [19]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=3e-5)

In [20]:
import time

start = time.time()
for i, batch in enumerate(train_dataloader):
    if i >= 20:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

elapsed = time.time() - start
per_batch = elapsed / 20
total_batches = len(train_dataloader)
print(f"{per_batch:.3f}s/batch, estimated epoch time: {(per_batch * total_batches)/60:.1f} min")

0.262s/batch, estimated epoch time: 49.1 min


In [21]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"  Batch {i}/{len(dataloader)} — loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

In [22]:
@torch.no_grad()
def evaluate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

    return total_loss / len(dataloader)

In [25]:
NUM_EPOCHS = 2

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss = train_epoch(model, train_dataloader, optimizer, device)
    val_loss = evaluate_epoch(model, val_dataloader, device)

    print(f"Train loss: {train_loss:.4f}, Validation loss: {val_loss:.4f}")

Epoch 1/2
  Batch 0/11250 — loss: 3.8472
  Batch 50/11250 — loss: 3.4281
  Batch 100/11250 — loss: 3.7613
  Batch 150/11250 — loss: 2.6081
  Batch 200/11250 — loss: 2.3664
  Batch 250/11250 — loss: 2.3254
  Batch 300/11250 — loss: 2.7194
  Batch 350/11250 — loss: 3.5616
  Batch 400/11250 — loss: 2.7072
  Batch 450/11250 — loss: 2.3811
  Batch 500/11250 — loss: 2.3043
  Batch 550/11250 — loss: 2.3089
  Batch 600/11250 — loss: 2.9188
  Batch 650/11250 — loss: 3.5229
  Batch 700/11250 — loss: 2.6391
  Batch 750/11250 — loss: 3.6221
  Batch 800/11250 — loss: 3.6302
  Batch 850/11250 — loss: 3.4107
  Batch 900/11250 — loss: 2.5364
  Batch 950/11250 — loss: 2.9513
  Batch 1000/11250 — loss: 3.5715
  Batch 1050/11250 — loss: 3.7395
  Batch 1100/11250 — loss: 2.8489
  Batch 1150/11250 — loss: 3.1846
  Batch 1200/11250 — loss: 2.5366
  Batch 1250/11250 — loss: 2.9979
  Batch 1300/11250 — loss: 2.7890
  Batch 1350/11250 — loss: 2.1958
  Batch 1400/11250 — loss: 2.9495
  Batch 1450/11250 — loss: 

In [26]:
predictions = []
references = []

for i in range(len(test_subset["hi"])):
    sentence = test_subset["hi"][i]
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = model.generate(**inputs, max_length=80, num_beams=4, early_stopping=True)
    translated_text = tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
    predictions.append(translated_text)
    references.append(test_subset["en"][i])

In [27]:
print(test_subset["hi"][0])
print(predictions[0])
print(references[0])

उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर बिलकुल भी भरोसा नहीं है।
He has no confidence in death in the manner of the CEO.
They never believed he died the way the sheriff concluded.


In [28]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.2 MB/s eta 0:00:00


In [29]:
import sacrebleu
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"Fine-tuned HI→EN BLEU: {bleu.score:.2f}")

Fine-tuned HI→EN BLEU: 14.98


In [30]:
model.save_pretrained("marian_hi_en_finetuned")
tokenizer.save_pretrained("marian_hi_en_finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('marian_hi_en_finetuned/tokenizer_config.json',
 'marian_hi_en_finetuned/vocab.json',
 'marian_hi_en_finetuned/source.spm',
 'marian_hi_en_finetuned/target.spm',
 'marian_hi_en_finetuned/added_tokens.json')

In [32]:
import json

marian_finetuned_hi_en_results = {
    "model": "MarianMT HI-EN, fine-tuned (2 epochs, manual loop)",
    "bleu_score": bleu.score,
    "baseline_bleu_score": 9.85,  # Day 5's zero-shot number
}

with open("marian_finetuned_hi_en_results.json", "w") as f:
    json.dump(marian_finetuned_hi_en_results, f, indent=2, ensure_ascii=False)